# **IV. Análisis de variables categóricas**

## Objetivo
Este notebook tiene el fin de analizar la distribución de frecuencias de todas las variables categóricas presentes en el dataset de episodios oncológicos de GRD (2019-2024), generando matrices de frecuencias y porcentajes.

## Proceso
1. **Carga y estandarización**: Carga los archivos los CSV anuales de los episodios oncológicos de GRD y estandariza los nombres de columnas
2. **Análisis de distribución**: Genera matrices de frecuencias absolutas y relativas (%) por año
3. **Evaluación de cardinalidad**: Justificar el uso o descarte de las variables para la predicción de severidad, mortalidad o consumo de recursos de los episodios.

In [29]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================

import pandas as pd
import os
import warnings # Para suprimir advertencias durante la carga de datos
# Suprimir advertencias sobre cambios de tipo de dato (dtype)
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# Ruta de datos oncológicos sin procesar
carpeta = "../../Datos/Datos oncológicos (sin procesar)"

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]

# Mapeo de equivalencias para estandarizar nombres de columnas inconsistentes entre años
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",       # Corrige corrupción de encoding (BOM UTF-8)
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"     # Renombra a estándar de proyecto
}

In [40]:
# ============================================================================
# UTILIDADES: Funciones de Limpieza y Normalización
# ============================================================================

import unicodedata
import re

def normalizar_texto(texto):
    """
    - Descripción: Normaliza texto eliminando acentos y caracteres diacríticos.
                   Intenta convertir de latin1 a UTF-8 para manejar encoding mixtos.
    - Entrada: texto (str o None): Texto a normalizar
    - Salida: str o None: Texto normalizado sin acentos, o None si entrada es None
    """
    if texto is None:
        return None
    
    # Intentar corrección de encoding (encoding mixtos)
    try:
        texto = texto.encode("latin1").decode("utf-8")
    except:
        pass

    # Aplicar normalización NFD y eliminar diacríticos
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    return texto


def obtener_subcarpeta(columna):
    """
    - Descripción: Asigna a las variables las subcarpetas donde se guardarán sus frecuencias según su tipo o contenido.
                   Los servicios de traslado se guardan en "servicios_traslado"
                   Las variables neonatales se guardan en "neonatal"
                   Los diagnósticos se guardan en "diagnosticos"
                   Los procedimientos se guardan en "procedimientos"
                   El resto de variables se guardan en la raíz de la carpeta de frecuencias
    - Entrada: columna (str): Nombre de la columna/variable
    - Salida: str: Nombre de la subcarpeta destino (ej: "diagnosticos", "procedimientos")
    """
    # Detecta variables de servicios de traslado (SERVICIOTRASLADO1, SERVICIOTRASLADO2, etc.)
    if re.match(r"^SERVICIOTRASLADO\d+$", columna):
        return "servicios_traslado"
    # Detecta variables neonatales (recién nacido)
    if re.match(r"^(CONDICIONDEALTANEONATO\d+|PESORN\d+|SEXORN\d+|RN\d+ESTADO)$", columna):
        return "neonatal"
    # Detecta diagnósticos (DIAGNOSTICO1, DIAGNOSTICO2, etc.)
    if re.match(r"^DIAGNOSTICO\d+$", columna):
        return "diagnosticos"
    # Detecta procedimientos (PROCEDIMIENTO1, PROCEDIMIENTO2, etc.)
    if re.match(r"^PROCEDIMIENTO\d+$", columna):
        return "procedimientos"
    # Variables genéricas sin clasificación específica
    return ""

def homologar_categorias(col, nombre_columna):
    """
    - Descripción: función que homologa categorías que significan lo mismo (equivalencias semánticas).
                   Reemplaza variantes de la misma categoría por un valor estándar (reduciendo fragmentación de datos)
    - Entradas:
        - col (pd.Series): Serie con categorías a homologar
        - nombre_columna (str): Nombre de la columna (para reglas específicas)
    - Salida:
        pd.Series: Serie con categorías homologadas
    """
    # Mapeo específico para la variable ETNIA
    if nombre_columna == "ETNIA":
        mapa = {
            "NINGUNA": "NINGUNO",
            "NINGUN": "NINGUNO",
            "SIN INFORMACION": "NINGUNO",    # Opcional: variantes alternativas
            "NO APLICA": "NINGUNO"
        }
        col = col.replace(mapa)

    return col

def distribucion_categorica_matriz(carpeta, archivos, mapa_columnas, columna):
    """
    - Descripción: Genera matriz de distribución de frecuencias para una variable categórica (2019-2024).
                   Itera sobre los archivos anuales, limpia y normaliza los valores, 
                   homologa categorías equivalentes, maneja nulos reales vs strings basura, 
                   y genera una matriz con frecuencias absolutas y porcentajes por año. 
                   Finalmente, exporta el resultado a CSV organizado por subcarpeta 
                   según el tipo de variable.
    
    - Entradas:
        - carpeta (str): Ruta carpeta datos oncológicos
        - archivos (list): Lista de nombres de archivos CSV (2019-2024)
        - mapa_columnas (dict): Diccionario mapeo nombres inconsistentes
        - columna (str): Nombre de variable categórica a analizar

    - Salida: pd.DataFrame: Matriz con frecuencias y porcentajes (categorías por años)
    """
    resultados = []
    categorias_globales = set()

    # Procesar cada año
    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año_archivo = int(archivo[-8:-4])  # Extrae año (ej: 2019)

        # Cargar CSV con estandarización de columnas
        df = pd.read_csv(ruta, low_memory=False).rename(columns=mapa_columnas)

        # Saltar si columna no existe en este año
        if columna not in df.columns:
            continue

        col_original = df[columna]

        # Identificar nulos reales (NaN de pandas) antes de conversiones
        mask_null_real = col_original.isna()

        col = col_original.copy().astype("object")

        # LIMPIEZA: aplicar solo a valores no nulos
        col[~mask_null_real] = (
            col[~mask_null_real]
            .astype(str)          # Convertir a string
            .str.strip()          # Remover espacios
            .apply(normalizar_texto)  # Normalizar acentos
            .str.upper()          # Convertir a mayúsculas
        )

        # Homologar categorías equivalentes
        col = homologar_categorias(col, columna)

        col = col.astype("object") # Mantener como object para categorías mixtas (strings + "NULOS")

        # Mapear strings basura a "NULOS"
        col = col.mask(col.isin(["", "NAN", "NONE", "NULL"]), "NULOS")

        col = col.astype("string")

        col[mask_null_real] = "NULOS"

        # Contar frecuencias de categorías
        conteo = col.value_counts()

        # Agregar categorías al conjunto global (para estadísticas)
        categorias_globales.update(conteo.index.tolist())

        # Registrar cada (categoría, año, frecuencia)
        for categoria, freq in conteo.items():
            resultados.append({
                "Categoría": categoria,
                "Año": año_archivo,
                "Frecuencia": freq
            })

    # Convertir resultados a DataFrame
    df_resultado = pd.DataFrame(resultados)

    # =====================================================================
    # GENERAR MATRIZ PIVOTE: Categorías × Años
    # =====================================================================
    matriz = df_resultado.pivot_table(
        index="Categoría",
        columns="Año",
        values="Frecuencia",
        fill_value=0
    )

    # Ordenar años cronológicamente (izquierda a derecha)
    matriz = matriz.sort_index(axis=1)

    # Ordenar por total descendente (categorías más frecuentes primero)
    matriz["TOTAL"] = matriz.sum(axis=1)
    matriz = matriz.sort_values("TOTAL", ascending=False)
    matriz = matriz.drop(columns="TOTAL")

    # Asegurar tipos numéricos para cálculos
    matriz = matriz.fillna(0).astype(int)

    # =====================================================================
    # CALCULAR PORCENTAJES Y COMBINAR CON FRECUENCIAS
    # =====================================================================
    # Calcular porcentajes por columna (año)
    matriz_pct = matriz.div(matriz.sum(axis=0), axis=1) * 100

    # Crear matriz final con formato "Frecuencia (Porcentaje%)"
    matriz_final = matriz.copy().astype(str)

    for col_año in matriz.columns:
        matriz_final[col_año] = (
            matriz[col_año].astype(int).astype(str)
            + " (" + matriz_pct[col_año].round(1).astype(str) + "%)"
        )

    # Limpiar metadatos de índices
    matriz_final.columns.name = None
    matriz_final.index.name = None

    # =====================================================================
    # CALCULAR Y REPORTAR CARDINALIDAD
    # =====================================================================
    # Total de categorías únicas (excluyendo NULOS/DESCONOCIDO)
    categorias_limpias = sorted([c for c in categorias_globales if c not in ["NULOS", "DESCONOCIDO"]])
    print(f"\n✓ Cardinalidad {columna}: {len(categorias_limpias)} categorías únicas")
    print(f"  ({', '.join(categorias_limpias[:10])}{'...' if len(categorias_limpias) > 10 else ''})")

    # =====================================================================
    # GUARDAR RESULTADO EN CSV
    # =====================================================================
    # Determinar subcarpeta según tipo de variable
    subcarpeta = obtener_subcarpeta(columna)
    carpeta_salida = os.path.join("../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)", subcarpeta)
    os.makedirs(carpeta_salida, exist_ok=True)

    # Guardar con encoding UTF-8-SIG (Excel compatible)
    ruta_salida = os.path.join(carpeta_salida, f"{columna.lower()}_frecuencia.csv")
    matriz_final.to_csv(ruta_salida, encoding="utf-8-sig")

    print(f"  [{subcarpeta}] Guardado en: {ruta_salida}")

    return matriz_final

## **1. Variables categóricas objetivo:**

### **Mortalidad (tipo de alta):** Verdadera mortalidad del paciente. Condición de egreso o destino del paciente al finalizar su hospitalización.
- **Tipo de variable**: Objetivo (target).
- **Cardinalidad**: Tiene 2 categorías y la etiqueta "DESCONOCIDO" es marginal (0.0%).
- Es el Ground Truth (Verdad Fundamental) del desenlace del paciente. Como hay muchas categorías y el estudio tiene el fin de determinar si el paciente fallece o no, se va a binarizar (categorías fallecido o vivo para el resto de casos).

In [56]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPOALTA")


✓ Cardinalidad TIPOALTA: 11 categorías únicas
  (ALTA VOLUNTARIA, DERIVACION A OTROS CENTROS (CARCEL, HOGAR DE, DERIVACION INST. PRIVADA (COMPRA DE SERVICIOS, DERIVACION INST. PRIVADA (VOLUNTARIO), DERIVACION OTRO HOSPITAL DE LA RED NACIONAL, DERIVACION OTRO HOSPITAL DEL SERVICIO, DOMICILIO, FALLECIDO, FUGA DEL PACIENTE, HOSPITALIZACION DOMICILIARIA...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\tipoalta_frecuencia.csv


,2019,2020,2021,2022,2023,2024
DOMICILIO,41297 (89.3%),31275 (88.6%),34674 (89.3%),40036 (89.3%),45856 (88.6%),48506 (88.0%)
FALLECIDO,2793 (6.0%),2018 (5.7%),2019 (5.2%),2293 (5.1%),2725 (5.3%),3037 (5.5%)
HOSPITALIZACION DOMICILIARIA,474 (1.0%),556 (1.6%),804 (2.1%),1057 (2.4%),1411 (2.7%),1621 (2.9%)
DERIVACION OTRO HOSPITAL DEL SERVICIO,796 (1.7%),636 (1.8%),504 (1.3%),579 (1.3%),808 (1.6%),982 (1.8%)
DERIVACION OTRO HOSPITAL DE LA RED NACIONAL,437 (0.9%),401 (1.1%),368 (0.9%),379 (0.8%),460 (0.9%),478 (0.9%)
ALTA VOLUNTARIA,281 (0.6%),241 (0.7%),291 (0.7%),330 (0.7%),340 (0.7%),293 (0.5%)
DERIVACION INST. PRIVADA (COMPRA DE SERVICIOS,66 (0.1%),54 (0.2%),56 (0.1%),46 (0.1%),60 (0.1%),69 (0.1%)
"DERIVACION A OTROS CENTROS (CARCEL, HOGAR DE",37 (0.1%),53 (0.2%),65 (0.2%),53 (0.1%),65 (0.1%),63 (0.1%)
FUGA DEL PACIENTE,33 (0.1%),37 (0.1%),30 (0.1%),38 (0.1%),34 (0.1%),40 (0.1%)
DERIVACION INST. PRIVADA (VOLUNTARIO),27 (0.1%),22 (0.1%),25 (0.1%),16 (0.0%),25 (0.0%),22 (0.0%)


### **Severidad:** Severidad de la enfermedad del paciente
- **Tipo de variable**: Objetivo (target).
- **Cardinalidad**: Es la métrica oficial del agrupador GRD que representa el nivel de severidad de la enfermedad (Severity of Illness). Clasifica la extensión de la descompensación fisiológica del paciente o la pérdida de función de sus órganos en una escala discreta ordinal: 0 (Sin gravedad), 1 (Menor), 2 (Moderada) y 3 (Mayor).

| ID | Gravedad     |
|----|--------------|
| 0  | Sin gravedad |
| 1  | Menor        |
| 2  | Moderada     |
| 3  | Mayor        |

In [57]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_SEVERIDAD")


✓ Cardinalidad IR_29301_SEVERIDAD: 4 categorías únicas
  (0, 1, 2, 3)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\ir_29301_severidad_frecuencia.csv


,2019,2020,2021,2022,2023,2024
1,17550 (38.0%),13129 (37.2%),13749 (35.4%),14671 (32.7%),15115 (29.2%),15782 (28.6%)
2,14134 (30.6%),11482 (32.5%),12225 (31.5%),13797 (30.8%),16548 (32.0%),17206 (31.2%)
3,8817 (19.1%),7910 (22.4%),8318 (21.4%),10061 (22.4%),11881 (22.9%),13316 (24.2%)
0,5740 (12.4%),2773 (7.9%),4544 (11.7%),6297 (14.0%),8240 (15.9%),8805 (16.0%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),2 (0.0%)


## **2. Resto de variables categóricas:**

### **Sexo:** Sexo del paciente.
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: Tiene 2 categorías y la etiqueta "DESCONOCIDO" es marginal (0.0%).
- **Veredicto**: Mantener, porque tiene una cardinalidad perfecta (de 2) y una distribución muy equilibrada a lo largo de todos los años (aproximadamente 56% de mujeres y 44% de hombres). Y puede ser un factor demográfico crítico en la literatura oncológica para la predicción de mortalidad y severidad.

In [ ]:
# Ejecutar análisis para variable SEXO (distribución por año 2019-2024)
resultado_sexo = distribucion_categorica_matriz(carpeta, archivos, mapa_columnas, "SEXO")

# Mostrar matriz de frecuencias
print("\n" + "="*70)
print("MATRIZ DE DISTRIBUCIÓN: SEXO (2019-2024)")
print("="*70)
print(resultado_sexo)


 Cardinalidad SEXO: 2 (HOMBRE, MUJER)
 [] Archivo guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\sexo_frecuencia.csv


,2019,2020,2021,2022,2023,2024
MUJER,26037 (56.3%),19708 (55.8%),21740 (56.0%),25398 (56.7%),29342 (56.7%),31060 (56.4%)
HOMBRE,20202 (43.7%),15583 (44.2%),17096 (44.0%),19428 (43.3%),22441 (43.3%),24034 (43.6%)
DESCONOCIDO,2 (0.0%),3 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),17 (0.0%)


### **Etnia:** Etnia del paciente (pertenencia a algún pueblo originario).
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: Tiene 11 categorías. 
- **Veredicto**: Descartar, porque las categorías "NINGUNO" y "OTRO" concentran entre el 97% y 99% de los datos en todos los años. En cambio, las etnias específicas (como Mapuche) apenas superan el 1%. Esto significa que la variable casi no tiene varianza, lo que introducirá ruido a los modelos de aprendizaje automático y probablemente no aporte poder predictivo a estos. 

In [6]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ETNIA")


✓ Cardinalidad ETNIA: 11 categorías únicas
  (AYMARA, COLLA, DIAGUITA, KAWESQAR, LICAN ANTAI (ATACAMENO), MAPUCHE, NINGUNO, OTRO, QUECHUA, RAPA NUI (PASCUENSE)...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\etnia_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NINGUNO,33249 (71.9%),30756 (87.1%),37347 (96.2%),0 (0.0%),0 (0.0%),54090 (98.1%)
OTRO,12444 (26.9%),3934 (11.1%),946 (2.4%),44171 (98.5%),50953 (98.4%),0 (0.0%)
MAPUCHE,514 (1.1%),522 (1.5%),449 (1.2%),474 (1.1%),610 (1.2%),821 (1.5%)
RAPA NUI (PASCUENSE),3 (0.0%),0 (0.0%),13 (0.0%),86 (0.2%),82 (0.2%),79 (0.1%)
AYMARA,1 (0.0%),29 (0.1%),35 (0.1%),51 (0.1%),76 (0.1%),55 (0.1%)
KAWESQAR,29 (0.1%),37 (0.1%),29 (0.1%),4 (0.0%),0 (0.0%),8 (0.0%)
DIAGUITA,0 (0.0%),0 (0.0%),7 (0.0%),13 (0.0%),22 (0.0%),31 (0.1%)
YAGAN (YAMANA),0 (0.0%),9 (0.0%),9 (0.0%),5 (0.0%),19 (0.0%),7 (0.0%)
QUECHUA,0 (0.0%),4 (0.0%),0 (0.0%),12 (0.0%),5 (0.0%),7 (0.0%)
LICAN ANTAI (ATACAMENO),0 (0.0%),3 (0.0%),0 (0.0%),4 (0.0%),11 (0.0%),7 (0.0%)


### **Provincia:** Provincia de residencia del paciente. 
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: Tiene 57 categorías (una cantidad grande)
- **Veredicto**: Agrupar, en específico se puede mapear las 57 provincias a las 16 regiones de Chile, ya que es manejable para los algoritmos de aprendizaje automático generar 15 o 16 columnas nuevas con One-Hot Encoding (en vez de generar 57)

In [7]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"PROVINCIA")


✓ Cardinalidad PROVINCIA: 57 categorías únicas
  (AISEN, ANTARTICA CHILENA, ANTOFAGASTA, ARAUCO, ARICA, BIO-BIO, CACHAPOAL, CAPITAN PRAT, CARDENAL CARO, CAUQUENES...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\provincia_frecuencia.csv


,2019,2020,2021,2022,2023,2024
SANTIAGO,11344 (24.5%),8531 (24.2%),9451 (24.3%),11062 (24.7%),13007 (25.1%),13744 (24.9%)
CONCEPCION,3197 (6.9%),2629 (7.4%),3234 (8.3%),3783 (8.4%),4246 (8.2%),4312 (7.8%)
VALPARAISO,3042 (6.6%),1607 (4.6%),1410 (3.6%),2241 (5.0%),2908 (5.6%),2920 (5.3%)
CAUTIN,1993 (4.3%),1689 (4.8%),1646 (4.2%),1811 (4.0%),2162 (4.2%),2799 (5.1%)
CORDILLERA,2191 (4.7%),1579 (4.5%),1846 (4.8%),1847 (4.1%),2188 (4.2%),2253 (4.1%)
ELQUI,1421 (3.1%),1249 (3.5%),1293 (3.3%),1421 (3.2%),1825 (3.5%),1958 (3.6%)
VALDIVIA,1138 (2.5%),1135 (3.2%),1139 (2.9%),1386 (3.1%),1609 (3.1%),1660 (3.0%)
LLANQUIHUE,1445 (3.1%),1133 (3.2%),1167 (3.0%),1122 (2.5%),1416 (2.7%),1530 (2.8%)
BIO-BIO,1303 (2.8%),880 (2.5%),1036 (2.7%),1414 (3.2%),1407 (2.7%),1492 (2.7%)
TALCA,1155 (2.5%),990 (2.8%),1105 (2.8%),1177 (2.6%),1345 (2.6%),1422 (2.6%)


### **Comuna:** Comuna de residencia del paciente. 
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: Tiene 340 categorías, teniendo una fragmentación muy alta.
- **Veredicto**: Descartar, porque tiene su cardinalidad es excesivamente alta (340). Y si se aplica One-Hot Encoding a esta variable, se agregarían 340 nuevas columnas al conjunto de datos de entrenamiento. Dado que la comuna con más pacientes (Puente Alto) apenas representa aproximadamente el 4% de los datos, la inmensa mayoría de estas 340 columnas estarán llenas de ceros (matriz dispersa), lo que puede diluir el peso predictivo de otras variables clínicas más importantes.

In [8]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"COMUNA")


✓ Cardinalidad COMUNA: 340 categorías únicas
  (AISEN, ALGARROBO, ALHUE, ALTO BIOBIO, ALTO DEL CARMEN, ALTO HOSPICIO, ANCUD, ANDACOLLO, ANGOL, ANTOFAGASTA...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\comuna_frecuencia.csv


,2019,2020,2021,2022,2023,2024
PUENTE ALTO,2067 (4.5%),1418 (4.0%),1713 (4.4%),1752 (3.9%),2039 (3.9%),2120 (3.8%)
VALPARAISO,1790 (3.9%),918 (2.6%),902 (2.3%),1266 (2.8%),1489 (2.9%),1492 (2.7%)
MAIPU,1079 (2.3%),810 (2.3%),881 (2.3%),1234 (2.8%),1545 (3.0%),1531 (2.8%)
SANTIAGO,821 (1.8%),621 (1.8%),673 (1.7%),974 (2.2%),1268 (2.4%),1424 (2.6%)
VALDIVIA,751 (1.6%),741 (2.1%),763 (2.0%),1059 (2.4%),1153 (2.2%),1149 (2.1%)
...,...,...,...,...,...,...
COLCHANE,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),2 (0.0%),2 (0.0%)
LAGUNA BLANCA,0 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
GENERAL LAGOS,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
SAN GREGORIO,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


### **Nacionalidad:** Nacionalidad del paciente. 
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: Tiene 75 categorías, pero hay valores continentales (como Europa y América) mezclados con países.
- **Veredicto**: Binarizar, tiene una cardinalidad muy alta (75) y la categoría "Chile" acapara entre el 95.9% y 97.7% de los episodios. Por ende, se puede binarizar y transformar la columna a "es chileno" (1 para Chile, 0 para el resto).

In [9]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"NACIONALIDAD")


✓ Cardinalidad NACIONALIDAD: 75 categorías únicas
  (AFGANISTAN, AFRICA, ALEMANIA, AMERICA, AMERICA DEL SUR, ANTILLAS HOLANDESAS, ARGELIA, ARGENTINA, AUSTRALIA, AUSTRIA...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\nacionalidad_frecuencia.csv


,2019,2020,2021,2022,2023,2024
CHILE,44905 (97.1%),34495 (97.7%),37782 (97.3%),43411 (96.8%),49862 (96.3%),52871 (95.9%)
VENEZUELA (REPUBLICA BOLIVARIANA DE),248 (0.5%),275 (0.8%),437 (1.1%),615 (1.4%),840 (1.6%),1013 (1.8%)
PERU,140 (0.3%),161 (0.5%),178 (0.5%),253 (0.6%),352 (0.7%),394 (0.7%)
COLOMBIA,100 (0.2%),92 (0.3%),123 (0.3%),181 (0.4%),248 (0.5%),254 (0.5%)
BOLIVIA (ESTADO PLURINACIONAL DE),75 (0.2%),76 (0.2%),122 (0.3%),135 (0.3%),200 (0.4%),239 (0.4%)
...,...,...,...,...,...,...
HONDURAS,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
LESOTHO,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
INDIA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
INDONESIA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)


### **Previsión:** Previsión de salud del paciente. 
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: 14, aunque hay categorías repetidas.
- **Veredicto**: Agrupar, porque es una variable que actúa como indicador del nivel socioeconómico del paciente, aunque su cardinalidad es un poco alta, por ende, se puede reducir a 6 categorías: los 4 tipos de FONASA MAI (A, B, C, D) que concentran el 95% de los datos, "DESCONOCIDO" para incluir las categorías "NO IDENTIFICADA" y "NO CONSIGNADO"; y agrupar el resto en "FFAA_Y_ORDEN".

In [10]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"PREVISION")


✓ Cardinalidad PREVISION: 14 categorías únicas
  (CAJA DE PREVISION FFAA (SISA), CAPREDENA, DIPRECA, FONASA INSTITUCIONAL - (MAI) A, FONASA INSTITUCIONAL - (MAI) B, FONASA INSTITUCIONAL - (MAI) C, FONASA INSTITUCIONAL - (MAI) D, FONASA LIBRE ELECCION (FMLE_B), FONASA LIBRE ELECCION (FMLE_C), FONASA LIBRE ELECCION (FMLE_D)...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\prevision_frecuencia.csv


,2019,2020,2021,2022,2023,2024
FONASA INSTITUCIONAL - (MAI) B,24780 (53.6%),19363 (54.9%),21615 (55.7%),25659 (57.2%),29936 (57.8%),32048 (58.2%)
FONASA INSTITUCIONAL - (MAI) A,8178 (17.7%),6239 (17.7%),6166 (15.9%),6955 (15.5%),7364 (14.2%),7125 (12.9%)
FONASA INSTITUCIONAL - (MAI) D,7050 (15.2%),5281 (15.0%),5967 (15.4%),6629 (14.8%),7707 (14.9%),8354 (15.2%)
FONASA INSTITUCIONAL - (MAI) C,5088 (11.0%),3942 (11.2%),4720 (12.2%),5090 (11.4%),6308 (12.2%),7080 (12.8%)
FONASA LIBRE ELECCION (FMLE_B),371 (0.8%),96 (0.3%),64 (0.2%),105 (0.2%),112 (0.2%),130 (0.2%)
ISAPRE,158 (0.3%),87 (0.2%),84 (0.2%),104 (0.2%),99 (0.2%),86 (0.2%)
PARTICULAR,130 (0.3%),101 (0.3%),74 (0.2%),100 (0.2%),63 (0.1%),73 (0.1%)
FONASA LIBRE ELECCION (FMLE_D),184 (0.4%),51 (0.1%),31 (0.1%),48 (0.1%),58 (0.1%),46 (0.1%)
DIPRECA,43 (0.1%),54 (0.2%),57 (0.1%),63 (0.1%),69 (0.1%),88 (0.2%)
FONASA LIBRE ELECCION (FMLE_C),121 (0.3%),27 (0.1%),8 (0.0%),29 (0.1%),25 (0.0%),37 (0.1%)


### **Servicio de salud:** Red de salud pública (macro-zona administrativa) a la que pertenece o fue derivado el paciente.
- **Tipo de variable**: Demográfica.
- **Cardinalidad**: 31, aunque hay categorías equivalentes como "DESCONOCIDO", "NO APLICA" e "IGNORADO".
- **Veredicto**: Mantener, ya que tiene una cardinalidad manejable para aplicar One-Hot-Encoding (31), además, los distintos servicios de salud tienen diferentes infraestructuras, presupuestos oncológicos y protocolos de derivación a nivel nacional.

In [11]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIO_SALUD")


✓ Cardinalidad SERVICIO_SALUD: 31 categorías únicas
  (ACONCAGUA, ANTOFAGASTA, ARAUCANIA NORTE, ARAUCANIA SUR, ARAUCO, ARICA, ATACAMA, AYSEN, BIOBIO, CHILOE...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicio_salud_frecuencia.csv


,2019,2020,2021,2022,2023,2024
METROPOLITANO SURORIENTE,4348 (9.4%),3194 (9.0%),3707 (9.5%),3792 (8.5%),4396 (8.5%),4554 (8.3%)
METROPOLITANO OCCIDENTE,3335 (7.2%),2265 (6.4%),2701 (7.0%),3312 (7.4%),3648 (7.0%),3612 (6.6%)
DEL MAULE,2976 (6.4%),2409 (6.8%),2516 (6.5%),2768 (6.2%),3094 (6.0%),3517 (6.4%)
METROPOLITANO CENTRAL,2405 (5.2%),1845 (5.2%),2032 (5.2%),2925 (6.5%),3780 (7.3%),3894 (7.1%)
METROPOLITANO ORIENTE,2521 (5.5%),1699 (4.8%),2027 (5.2%),2330 (5.2%),2724 (5.3%),3079 (5.6%)
CONCEPCION,2140 (4.6%),1783 (5.1%),2114 (5.4%),2370 (5.3%),2734 (5.3%),2727 (4.9%)
VINA DEL MAR QUILLOTA,2608 (5.6%),1543 (4.4%),1294 (3.3%),2067 (4.6%),2817 (5.4%),2962 (5.4%)
COQUIMBO,2141 (4.6%),1675 (4.7%),1907 (4.9%),2115 (4.7%),2568 (5.0%),2688 (4.9%)
METROPOLITANO SUR,2152 (4.7%),1916 (5.4%),1922 (4.9%),1981 (4.4%),2228 (4.3%),2486 (4.5%)
VALDIVIA,1596 (3.5%),1517 (4.3%),1866 (4.8%),1975 (4.4%),2350 (4.5%),2536 (4.6%)


### **Tipo de procedencia:** Indica el origen del paciente antes de ingresar al hospital (por ejemplo, desde un servicio de emergencia, un consultorio o derivado desde otra institución).
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 18, aunque las categorías NO IDENTIFICADO y DESCONOCIDO significan lo mismo (nulos).
- **Veredicto**: Mantener (con agrupación), ya que es un dato administrativo base conocido al momento de la admisión y casi no tiene valores nulos (los desconocidos son cercanos al 0.0%). Sin embargo, su cardinalidad (18) es un poco alta para el modelado con One Hot Encoding, por lo que se puede agrupar en macro categorías (como Ambulatorio/Cesfam, Urgencia, Derivación Hospitalaria, Sector Privado y otros).

In [12]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_PROCEDENCIA")


✓ Cardinalidad TIPO_PROCEDENCIA: 18 categorías únicas
  (APS CONSULTORIO (CESFAM), APS URGENCIA (SAPU, SUR, SUC), CARDIOCIRUGIA PAGO GRD, CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP), CIRUGIA MAYOR AMBULATORIA (CMA), CONSULTA PRIVADA, ESTRATEGIA CRR, HOSPITALIZACION DIURNA, HOSPITALIZACION DOMICILIARIA, LISTA DE ESPERA...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\tipo_procedencia_frecuencia.csv


,2019,2020,2021,2022,2023,2024
"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO ADOS. ESP)",29236 (63.2%),20932 (59.3%),24671 (63.5%),28711 (64.0%),33022 (63.8%),35093 (63.7%)
SERVICIO EMERGENCIA (DOMICILIO),11531 (24.9%),9741 (27.6%),9675 (24.9%),10161 (22.7%),11919 (23.0%),12579 (22.8%)
OTROS HOSPITALES DE LA RED,2013 (4.4%),1850 (5.2%),1633 (4.2%),1839 (4.1%),2274 (4.4%),2554 (4.6%)
"APS URGENCIA (SAPU, SUR, SUC)",603 (1.3%),629 (1.8%),649 (1.7%),694 (1.5%),791 (1.5%),1053 (1.9%)
OTROS HOSPITALES RED NACIONAL,768 (1.7%),602 (1.7%),564 (1.5%),573 (1.3%),585 (1.1%),635 (1.2%)
CONSULTA PRIVADA,935 (2.0%),411 (1.2%),355 (0.9%),473 (1.1%),481 (0.9%),492 (0.9%)
"OTRAS INSTITUCIONES SALUD (CLINICAS PRIVADAS, DE REHABILITAC",433 (0.9%),431 (1.2%),470 (1.2%),412 (0.9%),495 (1.0%),527 (1.0%)
APS CONSULTORIO (CESFAM),362 (0.8%),326 (0.9%),259 (0.7%),280 (0.6%),266 (0.5%),323 (0.6%)
PLAN DE RESOLUCION LE,0 (0.0%),0 (0.0%),0 (0.0%),1098 (2.4%),611 (1.2%),2 (0.0%)
CIRUGIA MAYOR AMBULATORIA (CMA),235 (0.5%),164 (0.5%),330 (0.8%),303 (0.7%),313 (0.6%),269 (0.5%)


### **Tipo de ingreso:** Tipo de ingreso del paciente al episodio asistencial (programado o urgencia).
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 4, ya que hay categorías equivalentes como NO IDENTIFICADA y DESCONOCIDO, que son conceptualmente lo mismo (ausencia de información).
- **Veredicto**: Mantener, ya que tiene una cardinalidad baja (4) y está fuertemente consolidada en dos clases principales (PROGRAMADA y URGENCIA). Además, es un dato base conocido al ingresar.

In [13]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_INGRESO")


✓ Cardinalidad TIPO_INGRESO: 4 categorías únicas
  (NO IDENTIFICADA, OBSTETRICA, PROGRAMADA, URGENCIA)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\tipo_ingreso_frecuencia.csv


,2019,2020,2021,2022,2023,2024
PROGRAMADA,30292 (65.5%),21737 (61.6%),25226 (65.0%),30443 (67.9%),35019 (67.6%),36667 (66.5%)
URGENCIA,15913 (34.4%),13501 (38.3%),13569 (34.9%),14329 (32.0%),16696 (32.2%),18328 (33.3%)
OBSTETRICA,30 (0.1%),56 (0.2%),41 (0.1%),52 (0.1%),68 (0.1%),108 (0.2%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),3 (0.0%),1 (0.0%),8 (0.0%)
NO IDENTIFICADA,6 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Especialidad médica:**  Especialidad médica principal encargada del tratamiento del paciente durante su episodio hospitalario.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 74, aunque los datos revelan un cambio a partir de 2020. Ya que categorías de 2019 como CIRUGIA TORAX, CIRUGIA COLOPROCTOLOGICA y MEDICINA GENERAL desaparecen y son reemplazadas desde 2020 por CIRUGIA DE TORAX, COLOPROCTOLOGIA y MEDICO GENERAL. Por ello, hay que unificar categorías.
- **Veredicto**: Unificar, se puede crear un diccionario para fusionar las especialidades equivalentes y agruparlas en unas 10 o 15 macro especialidades (como Cirugías, Medicina Interna, Cuidados Intensivos, Oncología, Ginecología/Urología, entre otras).

In [14]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ESPECIALIDAD_MEDICA")


✓ Cardinalidad ESPECIALIDAD_MEDICA: 74 categorías únicas
  (ANATOMIA PATOLOGICA, ANESTESIOLOGIA, CARDIOLOGIA, CIRUGIA CABEZA CUELLO Y PLASTICA MAXILO FACIAL, CIRUGIA CARDIOVASCULAR, CIRUGIA COLOPROCTOLOGICA, CIRUGIA DE CABEZA, CUELLO Y MAXILOFACIAL, CIRUGIA DE TORAX, CIRUGIA DIGESTIVA, CIRUGIA GENERAL...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\especialidad_medica_frecuencia.csv


,2019,2020,2021,2022,2023,2024
CIRUGIA GENERAL,16908 (36.6%),12686 (35.9%),13314 (34.3%),14761 (32.9%),15999 (30.9%),16394 (29.7%)
MEDICINA INTERNA,5939 (12.8%),4956 (14.0%),4845 (12.5%),5307 (11.8%),6345 (12.3%),6944 (12.6%)
OBSTETRICIA Y GINECOLOGIA,5620 (12.2%),4574 (13.0%),4905 (12.6%),5611 (12.5%),6475 (12.5%),7107 (12.9%)
UROLOGIA,4791 (10.4%),3645 (10.3%),4114 (10.6%),4894 (10.9%),5883 (11.4%),6151 (11.2%)
ONCOLOGIA MEDICA,2286 (4.9%),1352 (3.8%),1597 (4.1%),1963 (4.4%),2060 (4.0%),2269 (4.1%)
...,...,...,...,...,...,...
ODONTOPEDIATRIA,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
NEFROLOGIA PEDIATRICO,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
GENETICA CLINICA,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
INFECTOLOGIA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Tipo de actividad:** Tipo de actividad asociada al episodio (ej. hospitalización clásica vs. cirugía ambulatoria).
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 4, aunque se puede agrupar, ya que las categorías HOSPITALIZACION DIURNA y HOSPITALIZACION EN URGENCIA desaparecieron completamente después de 2019, se pueden agrupar con la categoría HOSPITALIZACION. 
- **Veredicto**: Mantener (con precaución), ya que esta variable puede tener alto riesgo de fuga (pues la actividad puede cambiar durante la evolución del paciente).

In [15]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"TIPO_ACTIVIDAD")


✓ Cardinalidad TIPO_ACTIVIDAD: 4 categorías únicas
  (CIRUGIA MAYOR AMBULATORIA (CMA), HOSPITALIZACION, HOSPITALIZACION DIURNA, HOSPITALIZACION EN URGENCIA)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\tipo_actividad_frecuencia.csv


,2019,2020,2021,2022,2023,2024
HOSPITALIZACION,39268 (84.9%),32521 (92.1%),34292 (88.3%),38530 (86.0%),43544 (84.1%),46306 (84.0%)
CIRUGIA MAYOR AMBULATORIA (CMA),3292 (7.1%),2773 (7.9%),4544 (11.7%),6297 (14.0%),8240 (15.9%),8805 (16.0%)
HOSPITALIZACION DIURNA,2445 (5.3%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
HOSPITALIZACION EN URGENCIA,1236 (2.7%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Servicio de ingreso:** servicio clínico específico donde el paciente es admitido físicamente al iniciar su episodio.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 78 categorías, aunque existen múltiples redundancias por digitación. Por ejemplo, UTI ADULTOS vs. UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO; o AREA MEDICA ADULTO CUIDADOS BASICOS vs. AREA MEDICO-QUIRURGICO CUIDADOS BASICOS.
- **Veredicto**: Agrupar, este dato es conocido inmediatamente al momento de la admisión, pero al tener una cardinalidad muy alta (de 78), se debe agrupar por nivel de complejidad: como UCI/UTI, Pabellón Quirúrgico, Cuidados Medios/Básicos, y Emergencia.

In [16]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIOINGRESO")


✓ Cardinalidad SERVICIOINGRESO: 78 categorías únicas
  (AGUDOS CIRUGIA, AGUDOS MEDICINA, AREA DE TRANSPLANTE ADULTO, AREA DE TRANSPLANTE INFANTIL, AREA MEDICA, AREA MEDICA ADULTO CUIDADOS BASICOS, AREA MEDICA ADULTO CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICO CUIDADOS BASICOS, AREA MEDICO-QUIRURGICO CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICOPEDIATRICA CUIDADOS BASICOS...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicioingreso_frecuencia.csv


,2019,2020,2021,2022,2023,2024
CIRUGIA,13982 (30.2%),10315 (29.2%),10055 (25.9%),11091 (24.7%),11456 (22.1%),11054 (20.1%)
GINECOLOGIA,4921 (10.6%),3942 (11.2%),4388 (11.3%),5057 (11.3%),5635 (10.9%),5700 (10.3%)
MEDICINA,4905 (10.6%),4149 (11.8%),3727 (9.6%),3880 (8.7%),4518 (8.7%),5265 (9.6%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),2803 (6.1%),1830 (5.2%),2967 (7.6%),3876 (8.6%),4888 (9.4%),5634 (10.2%)
UROLOGIA,2269 (4.9%),1708 (4.8%),1791 (4.6%),2382 (5.3%),2474 (4.8%),2376 (4.3%)
...,...,...,...,...,...,...
UNIDAD DE CUIDADOS INTENSIVOS NEONATOLOGIA,0 (0.0%),2 (0.0%),1 (0.0%),2 (0.0%),3 (0.0%),1 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO NEUROLOGIA,0 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),2 (0.0%)
NEUROPSIQUIATRIA INFANTIL,0 (0.0%),2 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%)
CIRUGIA PLASTICA-QUEMADOS ADULTOS,1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Servicios de traslado:** Registran cronológicamente las unidades o servicios clínicos específicos a los que el paciente fue trasladado durante la evolución de su episodio de hospitalización (después de su ingreso).
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: Entre 27 y 80 categorías. Además, existen múltiples redundancias por digitación y cambios de nomenclatura a lo largo de los años (ej. UTI ADULTOS vs. UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO), y códigos numéricos sin sentido descriptivo (ej. 408, 411, 405).
- **Veredicto**: Calcular la variable derivada cantidad de traslados, pero descartar estas columnas completamente (desde la 1 a la 9). Ya que tienen demasiados nulos, pues los datos revelan que en la variable SERVICIOTRASLADO1, alrededor del 81% de los registros son nulos, y entre los servicios 2 al 9, el porcentaje de nulos van aumentando hasta el 99%. Si se intenta aplicar One-Hot Encoding a este bloque se agregarían cientos de columnas nuevas al dataset, compuestas casi en un 100% de ceros, lo cual sería una matriz altamente dispersa que diluiría el poder predictivo de las variables base seleccionadas como TIPO_INGRESO o SERVICIO_SALUD

In [17]:
for i in range(1, 10):
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"SERVICIOTRASLADO{i}")


✓ Cardinalidad SERVICIOTRASLADO1: 80 categorías únicas
  (408, AGUDOS CIRUGIA, AGUDOS MEDICINA, AREA DE TRANSPLANTE ADULTO, AREA DE TRANSPLANTE INFANTIL, AREA MEDICA, AREA MEDICA ADULTO CUIDADOS BASICOS, AREA MEDICA ADULTO CUIDADOS MEDIOS, AREA MEDICA PEDIATRICA CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICO CUIDADOS BASICOS...)
  [servicios_traslado] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicios_traslado\serviciotraslado1_frecuencia.csv

✓ Cardinalidad SERVICIOTRASLADO2: 74 categorías únicas
  (408, AGUDOS CIRUGIA, AGUDOS MEDICINA, AREA DE TRANSPLANTE ADULTO, AREA DE TRANSPLANTE INFANTIL, AREA MEDICA, AREA MEDICA ADULTO CUIDADOS BASICOS, AREA MEDICA ADULTO CUIDADOS MEDIOS, AREA MEDICA PEDIATRICA CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICO CUIDADOS BASICOS...)
  [servicios_traslado] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicios_traslado\serviciotraslado2_frecuencia.csv

✓ Cardinalidad SERVICIOTRASLADO3: 71 

In [19]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIOTRASLADO1")


✓ Cardinalidad SERVICIOTRASLADO1: 80 categorías únicas
  (408, AGUDOS CIRUGIA, AGUDOS MEDICINA, AREA DE TRANSPLANTE ADULTO, AREA DE TRANSPLANTE INFANTIL, AREA MEDICA, AREA MEDICA ADULTO CUIDADOS BASICOS, AREA MEDICA ADULTO CUIDADOS MEDIOS, AREA MEDICA PEDIATRICA CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICO CUIDADOS BASICOS...)
  [servicios_traslado] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicios_traslado\serviciotraslado1_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,37783 (81.7%),27494 (77.9%),31444 (81.0%),36687 (81.8%),42360 (81.8%),44597 (80.9%)
UNIDAD DE TRATAMIENTO INTERMEDIO (UTI) (INDIFERENCIADO) ADULTO,1935 (4.2%),1320 (3.7%),997 (2.6%),1484 (3.3%),1850 (3.6%),2120 (3.8%)
CIRUGIA,1160 (2.5%),1263 (3.6%),955 (2.5%),997 (2.2%),1111 (2.1%),1334 (2.4%)
UNIDAD DE CUIDADOS INTENSIVOS ADULTO,795 (1.7%),634 (1.8%),682 (1.8%),669 (1.5%),705 (1.4%),857 (1.6%)
MEDICINA,710 (1.5%),701 (2.0%),505 (1.3%),605 (1.3%),708 (1.4%),777 (1.4%)
...,...,...,...,...,...,...
AREA MEDICO-QUIRURGICOPEDIATRICA CUIDADOS BASICOS,0 (0.0%),0 (0.0%),0 (0.0%),3 (0.0%),0 (0.0%),0 (0.0%)
PSIQUIATRIA CORTA ESTADIA,0 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
NEUROPSIQUIATRIA INFANTIL,0 (0.0%),2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA FISICA Y REHABILITACION,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


### **Servicio de alta:** 
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 80 categorías, aunque están las mismas redundancias que la variable SERVICIOINGRESO.
- **Veredicto**: Descartar, ya que la propuesta busca anticipar escenarios de riesgo utilizando herramientas predictivas "al inicio del episodio" para la planificación sanitaria. El servicio de alta es un evento que ocurre al final de la trayectoria del paciente. 

In [20]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"SERVICIOALTA")


✓ Cardinalidad SERVICIOALTA: 80 categorías únicas
  (AGUDOS CIRUGIA, AGUDOS MEDICINA, AREA DE TRANSPLANTE ADULTO, AREA DE TRANSPLANTE INFANTIL, AREA MEDICA, AREA MEDICA ADULTO CUIDADOS BASICOS, AREA MEDICA ADULTO CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICO CUIDADOS BASICOS, AREA MEDICO-QUIRURGICO CUIDADOS MEDIOS, AREA MEDICO-QUIRURGICOPEDIATRICA CUIDADOS BASICOS...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\servicioalta_frecuencia.csv


,2019,2020,2021,2022,2023,2024
CIRUGIA,14269 (30.9%),10955 (31.0%),10472 (27.0%),11412 (25.5%),11951 (23.1%),11758 (21.3%)
GINECOLOGIA,4954 (10.7%),3931 (11.1%),4345 (11.2%),5038 (11.2%),5562 (10.7%),5652 (10.3%)
MEDICINA,5211 (11.3%),4355 (12.3%),3803 (9.8%),4124 (9.2%),4879 (9.4%),5633 (10.2%)
UNIDAD DE RECUPERACION DE PABELLONES (CENTRAL Y CMA),2950 (6.4%),1971 (5.6%),3013 (7.8%),4009 (8.9%),5045 (9.7%),5501 (10.0%)
UROLOGIA,2340 (5.1%),1775 (5.0%),1834 (4.7%),2418 (5.4%),2489 (4.8%),2436 (4.4%)
...,...,...,...,...,...,...
MEDICINA FISICA Y REHABILITACION,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),3 (0.0%)
NEUROPSIQUIATRIA INFANTIL,0 (0.0%),1 (0.0%),0 (0.0%),1 (0.0%),1 (0.0%),0 (0.0%)
UNIDAD DE TRATAMIENTO INTERMEDIO NEUROLOGIA,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
CIRUGIA PLASTICA-QUEMADOS ADULTOS,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Datos neonatales:** Estas variables (que vienen en bloques de hasta 4 para registrar partos múltiples) indican los resultados de un recién nacido (RN) producto de un episodio de hospitalización obstétrica. Y registran la condición al momento del alta del bebé (vivo o fallecido) **(CONDICIONDEALTANEONATO{i})**, su sexo biológico **(SEXORN{i})** y códigos de estado de salud neonatal **(RN{i}ESTADO)**.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 2 categorías máximas en la condición de alta del bebé, 2 categorías máximas del sexo del bebé y 4 categorías máximas del estado de salud neonatal.
- **Veredicto**: Descartar, ya que la propuesta está delimitada a episodios clínicos con diagnóstico principal de neoplasias malignas (C00-C97), además, como se ve en las frecuencias, un parto de una paciente oncológica con cáncer activo como diagnóstico principal es un evento clínico extremadamente inusual (un outlier severo o caso atípico). Estas variables tienen gran cantidad de nulos (como la cantidad de alta que tiene casi un 100%), lo cual no es un aporte matemático para que los algoritmos puedan ajustar pesos y puede agregar ruido tras aplicar One-Hot-Encoding y mantener 12 columnas vacías o con valores 0 que no aporten al problema predictivo al realizar.

In [41]:
for i in range(1, 5):
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"CONDICIONDEALTANEONATO{i}")
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"SEXORN{i}")
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"RN{i}ESTADO")


✓ Cardinalidad CONDICIONDEALTANEONATO1: 2 categorías únicas
  (FALLECIDO, VIVO)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\condiciondealtaneonato1_frecuencia.csv

✓ Cardinalidad SEXORN1: 2 categorías únicas
  (HOMBRE, MUJER)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\sexorn1_frecuencia.csv

✓ Cardinalidad RN1ESTADO: 4 categorías únicas
  (0.0, 10.0, 6.0, 9.0)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\rn1estado_frecuencia.csv

✓ Cardinalidad CONDICIONDEALTANEONATO2: 0 categorías únicas
  ()
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\condiciondealtaneonato2_frecuencia.csv

✓ Cardinalidad SEXORN2: 1 categorías únicas
  (HOMBRE)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\sexorn2_frecuencia.csv

In [45]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"CONDICIONDEALTANEONATO1")


✓ Cardinalidad CONDICIONDEALTANEONATO1: 2 categorías únicas
  (FALLECIDO, VIVO)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\condiciondealtaneonato1_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,46238 (100.0%),35294 (100.0%),38836 (100.0%),44827 (100.0%),51784 (100.0%),55111 (100.0%)
VIVO,2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
FALLECIDO,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


In [46]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"SEXORN1")


✓ Cardinalidad SEXORN1: 2 categorías únicas
  (HOMBRE, MUJER)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\sexorn1_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,46238 (100.0%),35293 (100.0%),38834 (100.0%),44824 (100.0%),51783 (100.0%),55107 (100.0%)
MUJER,2 (0.0%),0 (0.0%),1 (0.0%),2 (0.0%),0 (0.0%),3 (0.0%)
HOMBRE,1 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%),1 (0.0%)


In [47]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"RN1ESTADO")


✓ Cardinalidad RN1ESTADO: 4 categorías únicas
  (0.0, 10.0, 6.0, 9.0)
  [neonatal] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\neonatal\rn1estado_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,0 (0.0%),22250 (63.0%),38834 (100.0%),44824 (100.0%),51783 (100.0%),55107 (100.0%)
0.0,46239 (100.0%),13043 (37.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
9.0,2 (0.0%),1 (0.0%),1 (0.0%),3 (0.0%),1 (0.0%),3 (0.0%)
10.0,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)
6.0,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Diagnósticos:** Diagnósticos del paciente durante ese episodio.

### **Diagnóstico principal de ingreso (diagnóstico 1):** Representa la condición clínica principal del ingreso hospitalario del paciente.
- **Tipo de variable**: Clínica.
- **Cardinalidad**: 466 categorías, aunque ya se ha agrupado en las macro categorías de tipos de cáncer (CIE-10) en la variable ``Categoría Cáncer`` al hacer el filtrado inicial de los episodios oncológicos.
- **Veredicto**: Agrupar, considerando las macro categorías del cáncer, ya que mantener 466 categorías únicas generaría una matriz demasiado dispersa al aplicar One-Hot Encoding. Al agrupar los códigos C00-C97 en 15 macro-categorías es exactamente la solución óptima, pues se reduce drásticamente la cardinalidad de 466 a 15.


### **Diagnósticos secundarios (diagnóstico 2 al diagnóstico 35):** Códigos CIE-10 adicionales que documentan condiciones preexistentes (comorbilidades como hipertensión o diabetes), metástasis en otros órganos, o complicaciones surgidas durante la hospitalización.
- **Tipo de variable**: Clínica.
- **Cardinalidad**: Alta, en algunos diagnósticos hay más de 4000 categorías.
- **Veredicto**: Descartar como columnas individuales y sintetizar en variables derivadas. Ya que hay columnas con más de 4.100 categorías únicas (como DIAGNOSTICO2). Y si se aplica OHE a estas 34 columnas, se crearían más de 100.000 nuevas columnas en el dataset y los algoritmos colapsarían computacionalmente y perderían su capacidad de generalización. Además, entre más alto es el número de la columna de diagnóstico, la cantidad de nulos va aumentando hasta alcanzar casi el 100% en el diagnóstico 35.

### **Variables derivadas que se pueden obtener desde los diagnósticos:**
- **Carga oncológica (numérica discreta):** consiste en contar cuántos códigos empiezan con la letra "C" (C00-C97) a lo largo de los 35 diagnósticos para un mismo paciente. Y puede representar la metástasis o la diseminación de la enfermedad.
- **Carga de comorbilidades (numérica discreta):** consiste en contar cuántos códigos no empiezan con la letra "C" en los diagnósticos del 2 al 35. (Por ejemplo: I10, E11.9, Z92.2). Por ejemplo, los pacientes no siempre fallecen por el tumor, pues las comorbilidades (diabetes, hipertensión) pueden complicar los cuadros, por ende puede ser una variable útil para predecir los riesgos. 
- **Comorbilidad principal (categórica agrupada):** Se extrae el primer diagnóstico secundario que no empiece con "C" (el Diagnóstico no oncológico principal), aunque requiere un ajuste, ya que puede incrementar una alta dimensionalidad, por ende se puede mapear sólo el Top 10 o Top 15 de enfermedades crónicas más frecuentes (como cardiovasculares, metabólicas, respiratorias), y agrupar el resto en la categoría "OTRA_COMORBILIDAD".

In [42]:
for i in range(1, 36):
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"DIAGNOSTICO{i}")


✓ Cardinalidad DIAGNOSTICO1: 466 categorías únicas
  (C00.0, C00.1, C00.2, C00.3, C00.4, C00.5, C00.6, C00.8, C00.9, C01...)
  [diagnosticos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\diagnosticos\diagnostico1_frecuencia.csv

✓ Cardinalidad DIAGNOSTICO2: 4110 categorías únicas
  (A02.0, A02.1, A02.9, A03.1, A03.3, A04.0, A04.1, A04.2, A04.4, A04.5...)
  [diagnosticos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\diagnosticos\diagnostico2_frecuencia.csv

✓ Cardinalidad DIAGNOSTICO3: 4163 categorías únicas
  (A02.0, A02.1, A03.9, A04.0, A04.2, A04.4, A04.5, A04.7, A04.8, A04.9...)
  [diagnosticos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\diagnosticos\diagnostico3_frecuencia.csv

✓ Cardinalidad DIAGNOSTICO4: 3946 categorías únicas
  (A02.0, A02.9, A03.8, A03.9, A04.0, A04.4, A04.5, A04.7, A04.8, A06.9...)
  [diagnosticos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2

In [48]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"DIAGNOSTICO1")


✓ Cardinalidad DIAGNOSTICO1: 466 categorías únicas
  (C00.0, C00.1, C00.2, C00.3, C00.4, C00.5, C00.6, C00.8, C00.9, C01...)
  [diagnosticos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\diagnosticos\diagnostico1_frecuencia.csv


,2019,2020,2021,2022,2023,2024
C50.9,4228 (9.1%),2788 (7.9%),3224 (8.3%),4348 (9.7%),4753 (9.2%),4660 (8.5%)
C73,1513 (3.3%),1238 (3.5%),1460 (3.8%),1873 (4.2%),2128 (4.1%),2126 (3.9%)
C16.9,1946 (4.2%),1413 (4.0%),1399 (3.6%),1699 (3.8%),1852 (3.6%),1907 (3.5%)
C61,1949 (4.2%),1214 (3.4%),1283 (3.3%),1595 (3.6%),2024 (3.9%),2147 (3.9%)
C20,1444 (3.1%),1100 (3.1%),1281 (3.3%),1471 (3.3%),1769 (3.4%),1942 (3.5%)
...,...,...,...,...,...,...
C86.3,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
C93.9,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
C88.3,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
C09.1,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


### **Procedimientos (procedimiento 1 al procedimiento 30):** Códigos que representan las intervenciones médicas, quirúrgicas, diagnósticas o terapéuticas que se le realizaron al paciente durante su hospitalización.
- **Tipo de variable**: hospitalaria.
- **Cardinalidad**: Desde 345 categorías en algunos procedimientos hasta 2274 categorías.
- **Veredicto**: Descartar completamente (eliminar las 30 columnas), ya que estos procedimientos ocurren durante la evolución de la estadía, por ende puede provocar fuga de datos. Además, hay procedimientos con más de 2000 categorías únicas (como procedimiento1 y procedimiento2) y al aplicar One-Hot Encoding (OHE) a estas 30 columnas, significaría inyectar más de 30.000 columnas nuevas al conjunto de datos, lo cual podría provocar sobreajuste masivo y colapso de memoria en el entrenamiento con los modelos de aprendizaje automático. Por último, tal como ocurre con los diagnósticos secundarios, la densidad de los datos cae drásticamente, alcanzando un 98% de nulos en el procedimiento 30, lo cual puede aportar más ruido que señal matemática en los algoritmos.

In [43]:
for i in range(1, 31):
    distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"PROCEDIMIENTO{i}")


✓ Cardinalidad PROCEDIMIENTO1: 2012 categorías únicas
  (0.1, 0.14, 0.17, 0.18, 0.19, 0.21, 0.22, 0.23, 0.24, 0.28...)
  [procedimientos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\procedimientos\procedimiento1_frecuencia.csv

✓ Cardinalidad PROCEDIMIENTO2: 2274 categorías únicas
  (0.01, 0.09, 0.1, 0.14, 0.17, 0.18, 0.19, 0.21, 0.23, 0.31...)
  [procedimientos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\procedimientos\procedimiento2_frecuencia.csv

✓ Cardinalidad PROCEDIMIENTO3: 2039 categorías únicas
  (0.01, 0.09, 0.1, 0.11, 0.12, 0.14, 0.17, 0.18, 0.19, 0.21...)
  [procedimientos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\procedimientos\procedimiento3_frecuencia.csv

✓ Cardinalidad PROCEDIMIENTO4: 1753 categorías únicas
  (0.01, 0.09, 0.11, 0.14, 0.17, 0.18, 0.19, 0.22, 0.23, 0.28...)
  [procedimientos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencia

In [49]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,f"PROCEDIMIENTO1")


✓ Cardinalidad PROCEDIMIENTO1: 2012 categorías únicas
  (0.1, 0.14, 0.17, 0.18, 0.19, 0.21, 0.22, 0.23, 0.24, 0.28...)
  [procedimientos] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\procedimientos\procedimiento1_frecuencia.csv


,2019,2020,2021,2022,2023,2024
85.23,2260 (4.9%),1813 (5.1%),2171 (5.6%),2952 (6.6%),3383 (6.5%),3412 (6.2%)
86.07,1750 (3.8%),1200 (3.4%),1998 (5.1%),2750 (6.1%),3865 (7.5%),3870 (7.0%)
99.25,1191 (2.6%),1242 (3.5%),1431 (3.7%),1708 (3.8%),1818 (3.5%),2163 (3.9%)
88.01,1440 (3.1%),1185 (3.4%),1109 (2.9%),1290 (2.9%),1650 (3.2%),1808 (3.3%)
57.49,1081 (2.3%),835 (2.4%),1067 (2.7%),1284 (2.9%),1491 (2.9%),1604 (2.9%)
...,...,...,...,...,...,...
78.62,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
78.63,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
78.75,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
78.81,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Especialidad de intervención:** Especialidad médica o quirúrgica del profesional que realizó una intervención durante la hospitalización.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 71 categorías, con problemas de consistencia temporal (como el cambio en 2020 de CIRUGIA COLOPROCTOLOGICA a COLOPROCTOLOGIA).
- **Veredicto**: Descartar, ya que puede provocar fuga de datos al ocurrir después del ingreso, además, tiene exceso de nulos (entre el 32% y 40% de los datos son nulos, porque no todos los pacientes oncológicos son intervenidos quirúrgicamente en todos sus ingresos). Finalmente, tiene muchas categorías (cardinalidad de 71) y se planea mantener la variable ``ESPECIALIDAD_MEDICA``, por lo que esta columna es redundante.

In [50]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"ESPECIALIDADINTERVENCION")


✓ Cardinalidad ESPECIALIDADINTERVENCION: 71 categorías únicas
  (ANATOMIA PATOLOGICA, ANESTESIOLOGIA, CARDIOLOGIA, CIRUGIA CABEZA CUELLO Y PLASTICA MAXILO FACIAL, CIRUGIA CARDIOVASCULAR, CIRUGIA COLOPROCTOLOGICA, CIRUGIA DE CABEZA, CUELLO Y MAXILOFACIAL, CIRUGIA DE TORAX, CIRUGIA DIGESTIVA, CIRUGIA GENERAL...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\especialidadintervencion_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,18816 (40.7%),13501 (38.3%),13065 (33.6%),14392 (32.1%),16607 (32.1%),18240 (33.1%)
CIRUGIA GENERAL,12747 (27.6%),10344 (29.3%),11871 (30.6%),13817 (30.8%),15072 (29.1%),15297 (27.8%)
UROLOGIA,4019 (8.7%),3120 (8.8%),3674 (9.5%),4263 (9.5%),5071 (9.8%),5433 (9.9%)
OBSTETRICIA Y GINECOLOGIA,3911 (8.5%),3173 (9.0%),3553 (9.1%),4346 (9.7%),4930 (9.5%),5003 (9.1%)
"CIRUGIA DE CABEZA, CUELLO Y MAXILOFACIAL",0 (0.0%),714 (2.0%),977 (2.5%),1466 (3.3%),1886 (3.6%),2379 (4.3%)
...,...,...,...,...,...,...
NEUROLOGIA PEDIATRICA,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA DEL DEPORTE,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
MEDICINA FAMILIAR,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
ENFERNEDADES RESPIRATORIAS PEDIATRICA,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **Uso de pabellón:** Indica si el paciente utilizó un quirófano (pabellón) y de qué tipo (central, urgencia, ambulatorio, etc.).
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 12 categorías, aunque hay algunas que no se encuentran en el diccionario de datos extraído desde la página de descarga de los datos GRD (probablemente estas categorías sean ruidos o errores).
- **Veredicto**: Mantener con precaución, ya que puede introducir fuga de datos, en especial para la predicción de la severidad. Por ejemplo, si un paciente usó un "Pabellón de Urgencia habilitado en UCI (6.0)", el modelo no está prediciendo la severidad, la está leyendo de las acciones que los médicos ya tomaron.

##### **Tabla (diccionario de datos extraído del sitio de "Fonasa - Datos abiertos")**:

| ID | Tipo de pabellón                         |
|----|------------------------------------------|
| 1  | Pabellón Central u Obstétrico           |
| 2  | Pabellón ambulatorio                    |
| 3  | Sala de procedimiento                   |
| 4  | Pabellón Hemodinamia                    |
| 5  | Pabellón de Urgencia                    |
| 6  | Pabellón de Urgencia habilitado en UCI  |
| 7  | H. de Campaña                           |
| 8  | Operativos Especiales                   |


In [51]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"USOSPABELLON")


✓ Cardinalidad USOSPABELLON: 12 categorías únicas
  (0.0, 1.0, 11.0, 2.0, 200.0, 249.0, 3.0, 4.0, 5.0, 6.0...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\usospabellon_frecuencia.csv


,2019,2020,2021,2022,2023,2024
1.0,26046 (56.3%),3 (0.0%),23248 (59.9%),26888 (60.0%),30071 (58.1%),31266 (56.7%)
NULOS,16803 (36.3%),35288 (100.0%),11527 (29.7%),13082 (29.2%),14785 (28.6%),16252 (29.5%)
2.0,2106 (4.6%),1 (0.0%),2711 (7.0%),3301 (7.4%),4739 (9.2%),4838 (8.8%)
3.0,1264 (2.7%),2 (0.0%),1194 (3.1%),1364 (3.0%),1887 (3.6%),2214 (4.0%)
4.0,8 (0.0%),0 (0.0%),63 (0.2%),85 (0.2%),157 (0.3%),255 (0.5%)
5.0,2 (0.0%),0 (0.0%),90 (0.2%),103 (0.2%),127 (0.2%),129 (0.2%)
6.0,0 (0.0%),0 (0.0%),0 (0.0%),4 (0.0%),15 (0.0%),134 (0.2%)
8.0,2 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),3 (0.0%),23 (0.0%)
0.0,6 (0.0%),0 (0.0%),3 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
11.0,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


### **IR_29301_COD_GRD:** código final del Grupo Relacionado por el Diagnóstico (GRD) asignado al paciente mediante el software agrupador al finalizar su estadía.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 795 categorías (cardinalidad muy alta).
- **Veredicto**: Descartar completamente al poder introducir data leakage, porque este código GRD es el resultado final del algoritmo de facturación de FONASA, calculado a partir de los diagnósticos, procedimientos y condición de egreso, y el peso GRD (variable objetivo de consumo de recursos) se extrae directamente de este código. Además, tiene 795 categorías únicas y al aplicar One-Hot Encoding podría crear casi 800 columnas muy dispersas.

In [52]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_COD_GRD")


✓ Cardinalidad IR_29301_COD_GRD: 795 categorías únicas
  (011012, 011013, 011101, 011102, 011103, 011111, 011112, 011113, 011122, 011123...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\ir_29301_cod_grd_frecuencia.csv


,2019,2020,2021,2022,2023,2024
161201,2123 (4.6%),1699 (4.8%),1767 (4.5%),2095 (4.7%),2148 (4.1%),2281 (4.1%)
92420,1713 (3.7%),720 (2.0%),1360 (3.5%),0 (0.0%),3392 (6.6%),0 (0.0%)
61202,1824 (3.9%),1500 (4.3%),1674 (4.3%),0 (0.0%),2113 (4.1%),0 (0.0%)
61201,1972 (4.3%),1544 (4.4%),1736 (4.5%),0 (0.0%),1809 (3.5%),0 (0.0%)
161202,929 (2.0%),719 (2.0%),831 (2.1%),1067 (2.4%),1325 (2.6%),1250 (2.3%)
...,...,...,...,...,...,...
082820,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
81303,0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
23130,1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
111121,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%)


### **IR_29301_MORTALIDAD:** probabilidad teórica de que el paciente fallezca.
- **Tipo de variable**: clínica.
- **Cardinalidad**: 4 categorías distintas.
- **Veredicto**: Descartar, ya que no sirve como predictor al ser un valor calculado al final del episodio usando los diagnósticos y procedimientos (lo que puede provocar fuga de datos) y no sirve como variable objetivo, ya que se cuenta con el dato real del desenlace del paciente en la variable TIPOALTA (agrupada en vivo o fallecido).

##### **Tabla (diccionario de datos extraído del sitio de "Fonasa - Datos abiertos")**:

| ID | Gravedad     |
|----|--------------|
| 0  | Sin gravedad |
| 1  | Menor        |
| 2  | Moderada     |
| 3  | Mayor        |

In [53]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"IR_29301_MORTALIDAD")


✓ Cardinalidad IR_29301_MORTALIDAD: 4 categorías únicas
  (0, 1, 2, 3)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\ir_29301_mortalidad_frecuencia.csv


,2019,2020,2021,2022,2023,2024
1,21201 (45.8%),16076 (45.5%),16737 (43.1%),18587 (41.5%),19952 (38.5%),20582 (37.3%)
3,10319 (22.3%),8973 (25.4%),9627 (24.8%),10862 (24.2%),13064 (25.2%),14401 (26.1%)
2,8981 (19.4%),7472 (21.2%),7928 (20.4%),9080 (20.3%),10528 (20.3%),11321 (20.5%)
0,5740 (12.4%),2773 (7.9%),4544 (11.7%),6297 (14.0%),8240 (15.9%),8805 (16.0%)
DESCONOCIDO,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),2 (0.0%)


### **Hospital de procedencia:** El nombre o código del hospital específico desde el cual el paciente fue derivado.
- **Tipo de variable**: Hospitalaria.
- **Cardinalidad**: 347 categorías (cardinalidad muy alta).
- **Veredicto**: Descartar, porque entre el 89% y el 92% de los registros son nulos (esto significa que 9 de cada 10 pacientes no vienen derivados de otro hospital específico, sino que ingresan directamente). Además, tiene una dimensionalidad muy alta (con 347 categorías) y es excesivo para aplicar One-Hot Encoding, además, se consideró agrupar otra variable que puede ser equivalente ``TIPO_PROCEDENCIA``.

In [ ]:
distribucion_categorica_matriz(carpeta, archivos, mapa_columnas,"HOSPPROCEDENCIA")


✓ Cardinalidad HOSPPROCEDENCIA: 347 categorías únicas
  (111351, 200559, 200696, CENTRO DE DIALISIS HEMOSUR (OSORNO), CENTRO DE DIALISIS Y ESPECIALIDADES MEDICAS PUERTO VARAS, CENTRO DE ENFERMEDADES RESPIRATORIAS INFANTILES JOSEFINA MARTINEZ (D), CENTRO DE REFERENCIA DE SALUD DE MAIPU, CENTRO MEDICO SAN JORGE RED DE SALUD UC CHRISTUS, CENTRO ONCOLOGICO DEL NORTE (CON), CLINICA ADVENTISTA...)
  [] Guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Frecuencias (categóricas)\hospprocedencia_frecuencia.csv


,2019,2020,2021,2022,2023,2024
NULOS,42063 (91.0%),31487 (89.2%),34707 (89.4%),40285 (89.9%),46382 (89.6%),50882 (92.3%)
HOSPITAL SAN PABLO (COQUIMBO),812 (1.8%),682 (1.9%),822 (2.1%),885 (2.0%),1056 (2.0%),999 (1.8%)
HOSPITAL CLINICO DE MAGALLANES DR. LAUTARO NAVARRO AVARIA,434 (0.9%),255 (0.7%),371 (1.0%),480 (1.1%),530 (1.0%),0 (0.0%)
HOSPITAL CARLOS VAN BUREN (VALPARAISO),386 (0.8%),251 (0.7%),217 (0.6%),263 (0.6%),357 (0.7%),379 (0.7%)
HOSPITAL SAN JOSE (CORONEL),204 (0.4%),187 (0.5%),138 (0.4%),155 (0.3%),183 (0.4%),204 (0.4%)
...,...,...,...,...,...,...
CLINICA SAN CARLOS DE APOQUINDO,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
CLINICA NUNOA,0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
CLINICA MUTUAL DE SEGURIDAD CCHC DE CALAMA,0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%)
CLINICA ENSENADA,0 (0.0%),0 (0.0%),0 (0.0%),1 (0.0%),0 (0.0%),0 (0.0%)
